# Lesson 10 — Decision Trees & K-Nearest Neighbours
**NSW Software Engineering Stage 6 | Software Automation**

---
### Two very different ways to think
So far we've used **regression** algorithms. Today we meet two completely different approaches:

* **Decision Trees** — ask a series of yes/no questions
* **K-Nearest Neighbours (KNN)** — find the most similar examples and copy their answer

**Your Goals for Today:**
1. Build and visualise a **Decision Tree** classifier.
2. Build a **KNN** classifier and understand the effect of `k`.
3. Compare both algorithms on the same dataset.

Take your time. Read the text, and click **Run** on each code box as you go down the page.

In [ ]:
# 👇 RUN THIS CELL FIRST 👇
!pip install numpy scikit-learn matplotlib --prefer-binary --quiet

import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.neighbors import KNeighborsClassifier
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import StandardScaler

print("✅ Setup Complete! Move to the next section.")

---
## Concept 1: Decision Trees — Like 20 Questions

A decision tree makes predictions by asking a series of **yes/no questions** about the features.

```
         Is petal_length < 2.5 cm?
              /            \
            YES              NO
             |                |
          SETOSA      Is petal_width < 1.75?
                           /        \
                         YES          NO
                          |            |
                     VERSICOLOR    VIRGINICA
```

Each question **splits** the data into purer groups. The AI automatically finds the **best questions** to ask.

**Key setting: `max_depth`** — how many questions the tree is allowed to ask.
Too deep = overfits. Too shallow = underfits.

In [ ]:
# 👇 RUN THIS CELL 👇

iris = load_iris()
X, y = iris.data, iris.target
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)

# Train a Decision Tree
dt = DecisionTreeClassifier(max_depth=3, random_state=42)
dt.fit(X_tr, y_tr)

acc = accuracy_score(y_te, dt.predict(X_te))
print(f"✅ Decision Tree trained!")
print(f"   Test Accuracy: {acc:.0%}")
print(f"\n📊 Feature Importances:")
for name, imp in zip(iris.feature_names, dt.feature_importances_):
    bar = '█' * int(imp * 25)
    print(f"   {name:30s} {imp:.3f} {bar}")

In [ ]:
# 👇 RUN THIS CELL 👇
# Visualise the tree

plt.figure(figsize=(14, 5))
plot_tree(dt,
          feature_names=iris.feature_names,
          class_names=iris.target_names,
          filled=True, rounded=True, fontsize=8)
plt.title("Decision Tree — Iris Classification (max_depth=3)")
plt.tight_layout(); plt.show()

print("\n=== Tree as Text ===")
print(export_text(dt, feature_names=iris.feature_names))

---
## Concept 2: K-Nearest Neighbours — Copy Your Neighbours

KNN is the simplest possible idea:
> *"To classify a new point, find the **k** most similar training points, and take a majority vote."*

**Steps:**
1. Pick a value of `k` (e.g., 5)
2. Measure the **distance** from the new point to every training point
3. Find the `k` closest points
4. Predict the **most common class** among them

**⚠️ Important:** KNN uses distances, so you MUST scale your features first!

In [ ]:
# 👇 RUN THIS CELL 👇

# Scale first — distances only make sense on the same scale
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_tr)
X_te_s = scaler.transform(X_te)

print(f"{'k':>4}  {'Train Acc':>12}  {'Test Acc':>12}  {'Note':>20}")
print("-" * 55)

for k in [1, 3, 5, 7, 11, 15]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_tr_s, y_tr)
    tr_acc = accuracy_score(y_tr, knn.predict(X_tr_s))
    te_acc = accuracy_score(y_te, knn.predict(X_te_s))
    note = " ← likely overfit" if tr_acc - te_acc > 0.06 else ""
    print(f"{k:>4}  {tr_acc:>12.0%}  {te_acc:>12.0%}  {note}")

---
## ✏️ Exercise 1: Build a Decision Tree for Insects

Fill in the `____` blanks to classify insect sex (0=female, 1=male) using a Decision Tree.

In [ ]:
# 👇 FILL IN THE BLANKS AND RUN THIS CELL 👇
import pandas as pd
df = pd.read_csv('insects.csv', sep='\t')

X_ins = df[['continent','latitude','wingsize']].values
y_ins = df['sex'].values

# STEP 1: Split 80/20
X_tr_i, X_te_i, y_tr_i, y_te_i = train_test_split(
    X_ins, y_ins, test_size=____, random_state=42   # TODO: fill in 0.2
)

# STEP 2: Create a DecisionTreeClassifier with max_depth=4
# TODO: Replace ____ with DecisionTreeClassifier(max_depth=4, random_state=42)
dt_ins = ____

# STEP 3: Train the tree
dt_ins.fit(____, ____)   # TODO: X_tr_i, y_tr_i

# STEP 4: Calculate accuracy
acc_ins = accuracy_score(____, dt_ins.predict(X_te_i))   # TODO: y_te_i

# ⬇️ DON'T EDIT BELOW THIS LINE — THIS IS THE AUTOGRADER ⬇️
print("=== Exercise 1 Results ===")
Xtr_c,Xte_c,ytr_c,yte_c = train_test_split(X_ins,y_ins,test_size=0.2,random_state=42)
correct_acc = accuracy_score(yte_c, DecisionTreeClassifier(max_depth=4,random_state=42).fit(Xtr_c,ytr_c).predict(Xte_c))
try:
    assert abs(acc_ins - correct_acc) < 0.03
    print(f"✅ Accuracy: {acc_ins:.0%}")
except: print(f"❌ Accuracy: got {acc_ins:.0%}, expected ~{correct_acc:.0%}")

print("\nFeature importances:")
feats = ['continent','latitude','wingsize']
for name, imp in zip(feats, dt_ins.feature_importances_):
    print(f"   {name}: {imp:.3f}")

---
## ✏️ Exercise 2: Decision Trees vs KNN — When to Use Each?

Replace `None` with `"decision_tree"` or `"knn"`.

In [ ]:
# 👇 FILL IN THE BLANKS AND RUN THIS CELL 👇

algorithm_quiz = {
    # Example:
    "A doctor needs to explain exactly WHY the model made its prediction": "decision_tree",

    # TODO: Fill in the 4 blanks:
    "You need to classify quickly after training — prediction speed matters": None,
    "Features are on very different scales and you forgot to scale them": None,
    "You want to visualise the rules the model learned as a diagram": None,
    "Your training set is tiny and you want a quick baseline model": None,
}

# ⬇️ DON'T EDIT BELOW THIS LINE — THIS IS THE AUTOGRADER ⬇️
correct_alg = {
    "A doctor needs to explain exactly WHY the model made its prediction": "decision_tree",
    "You need to classify quickly after training — prediction speed matters": "decision_tree",
    "Features are on very different scales and you forgot to scale them": "decision_tree",
    "You want to visualise the rules the model learned as a diagram": "decision_tree",
    "Your training set is tiny and you want a quick baseline model": "knn",
}
score = 0
print("=== Exercise 2 Results ===")
for q, your_ans in algorithm_quiz.items():
    if "doctor" in q: continue
    ans = correct_alg[q]
    if your_ans == ans:
        print(f"  ✅ '{ans}' — {q[:60]}")
        score += 1
    elif your_ans is None:
        print(f"  ⬜ Not answered")
    else:
        print(f"  ❌ You said '{your_ans}', correct is '{ans}'")
print(f"\nFinal Score: {score}/4")

---
## 🚀 EXTENSION CHALLENGES 🚀

### Exercise 3: Effect of max_depth on Decision Tree

Run the code below to see how `max_depth` affects training vs test accuracy. Then answer the question.

In [ ]:
# 👇 RUN AND ANSWER THE QUESTION 👇
iris = load_iris()
X3, y3 = iris.data, iris.target
Xtr3, Xte3, ytr3, yte3 = train_test_split(X3, y3, test_size=0.2, random_state=42)

depths = range(1, 12)
tr_accs, te_accs = [], []
for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=42)
    m.fit(Xtr3, ytr3)
    tr_accs.append(accuracy_score(ytr3, m.predict(Xtr3)))
    te_accs.append(accuracy_score(yte3, m.predict(Xte3)))

plt.figure(figsize=(8, 4))
plt.plot(depths, tr_accs, 'o-', color='steelblue', label='Train Accuracy')
plt.plot(depths, te_accs, 's-', color='tomato',    label='Test Accuracy')
plt.xlabel('max_depth'); plt.ylabel('Accuracy')
plt.title('Decision Tree: Depth vs Accuracy (Iris)')
plt.xticks(depths); plt.legend(); plt.tight_layout(); plt.show()

# TODO: Based on the plot, which max_depth is the sweet spot?
best_depth = None   # Replace None with the depth where test accuracy peaks
print(f"Your answer — best depth: {best_depth}")

### Exercise 4: The Debugging Challenge

The KNN code below has **TWO bugs** that will cause wrong results.
Find and fix them.

In [ ]:
# 👇 FIX THE TWO BUGS IN THIS CODE 👇
iris = load_iris()
X4, y4 = iris.data, iris.target
Xtr4, Xte4, ytr4, yte4 = train_test_split(X4, y4, test_size=0.2, random_state=0)

scaler4 = StandardScaler()
Xtr4_s = scaler4.fit_transform(Xtr4)

# BUG 1: We must scale the test data using the SAME scaler (not fit_transform again)
Xte4_s = scaler4.fit_transform(Xte4)    # ← wrong!

knn4 = KNeighborsClassifier(n_neighbors=5)
knn4.fit(Xtr4_s, ytr4)

# BUG 2: We should predict on scaled test data, not the raw unscaled data
acc4 = accuracy_score(yte4, knn4.predict(Xte4))   # ← wrong!

print(f"KNN Accuracy: {acc4:.0%}")
print("(Expected: ~97% when bugs are fixed)")

### Exercise 5: Head-to-Head Comparison 🏆

Run **both** a Decision Tree and a KNN classifier on the **insects dataset**.

For the Decision Tree, try depths 2, 3, 4 and pick the best.
For KNN, try k=3, 5, 7 and pick the best.

Print a comparison table and declare a winner.

In [ ]:
# 👇 WRITE ALL THE CODE YOURSELF IN THIS CELL 👇
import pandas as pd
df = pd.read_csv('insects.csv', sep='\t')
X5 = df[['continent','latitude','wingsize']].values
y5 = df['sex'].values

Xtr5, Xte5, ytr5, yte5 = train_test_split(X5, y5, test_size=0.2, random_state=42)

# Decision Tree: try depths 2, 3, 4 — print accuracy for each


# KNN: scale first, try k=3, 5, 7 — print accuracy for each


# Print which algorithm wins on this dataset




---
## ✅ Lesson 10 Complete!
You now know two more powerful AI algorithms — Decision Trees and KNN.

**Next up:** Lesson 11 — Neural Networks. We'll look inside the most powerful type of AI that powers ChatGPT, image recognition, and more.

---









































## 🔑 Answer Key

### Exercise 1
```python
train_test_split(X_ins, y_ins, test_size=0.2, ...)
dt_ins = DecisionTreeClassifier(max_depth=4, random_state=42)
dt_ins.fit(X_tr_i, y_tr_i)
acc_ins = accuracy_score(y_te_i, dt_ins.predict(X_te_i))
```

### Exercise 2
```
"classify quickly"               → "decision_tree"
"forgot to scale"                → "decision_tree"
"visualise rules as diagram"     → "decision_tree"
"tiny training set, quick model" → "knn"
```

### Exercise 3 — Best depth is usually 3 or 4 for Iris.

### Exercise 4 — Two bugs fixed:
```python
Xte4_s = scaler4.transform(Xte4)          # use transform, not fit_transform
acc4 = accuracy_score(yte4, knn4.predict(Xte4_s))   # predict on scaled data
```

### Exercise 5 — Sample answer:
```python
for d in [2,3,4]:
    m=DecisionTreeClassifier(max_depth=d).fit(Xtr5,ytr5)
    print(f"DT depth={d}: {accuracy_score(yte5,m.predict(Xte5)):.0%}")
sc=StandardScaler(); Xtr5s=sc.fit_transform(Xtr5); Xte5s=sc.transform(Xte5)
for k in [3,5,7]:
    m=KNeighborsClassifier(n_neighbors=k).fit(Xtr5s,ytr5)
    print(f"KNN k={k}: {accuracy_score(yte5,m.predict(Xte5s)):.0%}")
```
